# Export KPI — per-topic verification

Run the **Setup** cell first, then run each KPI cell one at a time and eyeball the
per-country result. Every KPI is written in plain pandas, straight from the
business-logic doc, so you can verify the logic independently of the app.

- Switch `SOURCE` in Setup between `"sheet"` (your live Google Sheet) and `"local"`
  (`data/test_data.xlsx`).
- All "count" KPIs are **SUM(Quantity)** (machines); the two "commitment changes"
  KPIs are **averages**; date KPIs are the **latest** date per country.
- The breakup spans **all months** (an order booked in a prior month can still be
  pending/overdue today). Set `BREAKUP_MONTH` in Setup to scope to one month.
- The last cell cross-checks every KPI against the app's `kpi_engine`.


## Setup — load data (all months), define helpers

In [ ]:
import pandas as pd
from datetime import date

pd.set_option("display.max_rows", 100)

DATE_FMT = "%d-%m-%Y"              # sheet dates are day-first, e.g. 05-07-2026
TODAY = pd.Timestamp(date.today())
BREAKUP_MONTH = None             # None = ALL months (pending/overdue span months); or e.g. "JULY"

# ---- load data:  "sheet" = live Google Sheet,  "local" = data/test_data.xlsx ----
SOURCE = "sheet"
try:
    from app.core.data_source import load
    df = load(SOURCE)
except Exception as exc:
    print("app loader unavailable, reading local xlsx instead:", exc)
    df = pd.read_excel("data/test_data.xlsx", dtype=str)

df = df.fillna("").astype(str)
print("rows:", len(df), "| months:", sorted(df["Month"].str.strip().str.upper().unique()))

# ---- scope (all months by default) + shared helpers ----
if BREAKUP_MONTH:
    scope = df[df["Month"].str.strip().str.upper() == BREAKUP_MONTH].copy()
else:
    scope = df.copy()            # all months: an old order can still be pending/overdue today
scope["Qty"] = pd.to_numeric(scope["Quantity"], errors="coerce").fillna(0)
countries = sorted(c for c in scope["Country"].str.strip().unique() if c)

def parse(col):
    "Parse a day-first date column of `scope` -> datetime (NaT if blank/bad)."
    return pd.to_datetime(scope[col], format=DATE_FMT, errors="coerce")

def blank_or_past(col):
    p = parse(col); return p.isna() | (p < TODAY)

def is_past(col):
    p = parse(col); return p.notna() & (p < TODAY)

def is_blank(col):
    return parse(col).isna()

def machines_where(mask, label):
    "SUM(Quantity) per country over rows matching `mask` (every country shown)."
    return (scope.loc[mask].groupby("Country")["Qty"].sum()
            .reindex(countries, fill_value=0).astype(int).rename(label))

def average(col, label):
    "AVG of a numeric column per country (over all the country's rows)."
    vals = pd.to_numeric(scope[col], errors="coerce").fillna(0)
    return vals.groupby(scope["Country"]).mean().reindex(countries).round(1).rename(label)

def latest(col, label):
    "MAX (latest) date per country, formatted back to DD-MM-YYYY."
    return (parse(col).groupby(scope["Country"]).max()
            .reindex(countries).dt.strftime(DATE_FMT).fillna("").rename(label))

def earliest(col, label):
    "MIN (earliest) date per country, formatted back to DD-MM-YYYY."
    return (parse(col).groupby(scope["Country"]).min()
            .reindex(countries).dt.strftime(DATE_FMT).fillna("").rename(label))

def worst_delay_days(mask, col, label):
    "Days from the oldest (min) masked date to today, per country — worst case, not avg."
    p = parse(col)
    def _worst(idx):
        vals = p[idx & mask].dropna()
        return int((TODAY - vals.min()).days) if len(vals) else 0
    return pd.Series({c: _worst(scope["Country"].str.strip() == c) for c in countries}, name=label)

df.head()

## KPI 1. Breakup — Customer

Display the unique countries in the report.

In [ ]:
# KPI 1 — Breakup / Customer: unique country names
pd.Series(countries, name="Country")

## KPI 2. Pending Orders

SUM(Quantity) where **Loading (Dispatched) Date** is blank OR **revision commitment of loading date** is earlier than today.

In [ ]:
# KPI 2 — Pending Orders
# Loading date blank OR revised loading commitment already past
mask = is_blank("Loading (Dispatched) Date") | is_past("revision commitment of loading date")
machines_where(mask, "Pending Orders")

## KPI 3. Overdue Breakup

SUM(Quantity) where **commitment of loading date** is blank OR earlier than today.

In [ ]:
# KPI 3 — Overdue Breakup
machines_where(blank_or_past("commitment of loading date"), "Overdue Breakup")

## KPI 4. No. of Days Delay from 1st Commitment

Among a country's overdue rows (Over Due Breakup mask), the oldest **Production Commitment Date** vs today, in days — the single worst delay, not an average.

In [ ]:
# KPI 4 — Days Delay from 1st Commitment (worst case, in days)
worst_delay_days(blank_or_past("commitment of loading date"), "Production Commitment Date", "Days Delay (worst)")

## KPI 5. New Committed Date

Latest (MAX) **revision commitment of loading date** per country.

In [ ]:
# KPI 5 — New Committed Date (latest)
latest("revision commitment of loading date", "New Committed Date")

## KPI 6. Container Expected Date

Latest (MAX) **Container Placement date** per country.

In [ ]:
# KPI 6 — Container Expected Date (latest)
latest("Container Placement date", "Container Expected Date")

## KPI 7A. Production Pending — No. of Machines

SUM(Quantity) where **Production Commitment Date** is blank, or given but **Production Commitment Revise Date** is earlier than today (same blank-or-revised-is-late shape as KPI 2).

In [ ]:
# KPI 7A — Production machines pending
machines_where(is_blank("Production Commitment Date") | is_past("Production Commitment Revise Date"), "Prdn Machines Pending")

## KPI 7B. Production Pending — Commitment Changes

AVG of **no of times commitment changes(prod)** per country.

In [ ]:
# KPI 7B — Production commitment changes (average)
average("no of times commitment changes(prod)", "Prdn Commitment Changes (avg)")

## KPI 7C. Production Overdue

SUM(Quantity) where **Production Commitment Revise Date** is earlier than today.

In [ ]:
# KPI 7C — Production Overdue
machines_where(is_past("Production Commitment Revise Date"), "Production Overdue")

## KPI 8A. Container Pending — No. of Machines

SUM(Quantity) where **Container Placement date** is blank, or it's overdue and **Container Revision Date** is still blank.

In [ ]:
# KPI 8A — Container machines pending
machines_where(
    is_blank("Container Placement date")
    | (is_past("Container Placement date") & is_blank("Container Revision Date")),
    "Container Machines Pending",
)

## KPI 8B. Container Pending — Commitment Changes

AVG of **no of comm container changes** per country.

In [ ]:
# KPI 8B — Container commitment changes (average)
average("no of comm container changes", "Container Commitment Changes (avg)")

## KPI 8C. Container Overdue

SUM(Quantity) where **Container Revision Date** is earlier than today.

In [ ]:
# KPI 8C — Container Overdue
machines_where(is_past("Container Revision Date"), "Container Overdue")

## KPI 9. Vessel Cut-Off

Earliest (MIN) **Vessel Cut-Off Date** per country.

In [ ]:
# KPI 9 — Vessel Cut-Off (earliest)
earliest("Vessel Cut-Off Date", "Vessel Cut-Off")

## KPI 10. Commercial Clearance no of Pending

SUM(Quantity) where **Commercial Clearance Status** is 'Pending' or blank.

In [ ]:
# KPI 10 — Commercial Clearance no of Pending
status = scope["Commercial Clearance Status"].str.strip().str.lower()
clearance_mask = status.isin(["pending", "", "nan", "none"])
machines_where(clearance_mask, "Commercial Clearance no of Pending")

## Appendix — Monthly summary (Opening / New / Total / Despatched / Balance)

Not part of the country breakup, but it's the top table of the report. Balance
carries forward month to month. **Despatched** = quantity whose Loading date is a
real date on/before today **and** not slipped into a later month than its bucket.

In [ ]:
MONTH_ORDER = ["MAY", "JUNE", "JULY"]
MONTH_NUM = {"JANUARY":1,"FEBRUARY":2,"MARCH":3,"APRIL":4,"MAY":5,"JUNE":6,
             "JULY":7,"AUGUST":8,"SEPTEMBER":9,"OCTOBER":10,"NOVEMBER":11,"DECEMBER":12}

d = df.copy()
d["Qty"] = pd.to_numeric(d["Quantity"], errors="coerce").fillna(0)
d["M"] = d["Month"].str.strip().str.upper()
loaded = pd.to_datetime(d["Loading (Dispatched) Date"], format=DATE_FMT, errors="coerce")
bucket = d["M"].map(MONTH_NUM)
dispatched = loaded.notna() & (loaded <= TODAY) & ((loaded.dt.month <= bucket) | bucket.isna())

rows, prev = [], 0
for m in MONTH_ORDER:
    inm = d["M"] == m
    new = int(d.loc[inm, "Qty"].sum())
    disp = int(d.loc[inm & dispatched, "Qty"].sum())
    total = prev + new
    balance = total - disp
    rows.append({"Month": m, "Opening": prev, "New": new, "Total": total,
                 "Despatched": disp, "Balance": balance})
    prev = balance
pd.DataFrame(rows).set_index("Month")

## Cross-check — plain pandas vs. the app's `kpi_engine`

Recomputes every count/average KPI in pandas and compares it, cell-for-cell,
against `compute_country_breakup`. Prints **MATCH** when they agree.

In [ ]:
from app.core.kpi_engine import compute_country_breakup

engine = pd.DataFrame(compute_country_breakup(df, month=BREAKUP_MONTH)).set_index("country")
status = scope["Commercial Clearance Status"].str.strip().str.lower()

mine = pd.concat([
    machines_where(is_blank("Loading (Dispatched) Date") | is_past("revision commitment of loading date"), "pending_orders"),
    machines_where(blank_or_past("commitment of loading date"), "over_due_breakup"),
    worst_delay_days(blank_or_past("commitment of loading date"), "Production Commitment Date", "days_delay"),
    machines_where(is_blank("Production Commitment Date") | is_past("Production Commitment Revise Date"), "prdn_machines_pending"),
    average("no of times commitment changes(prod)", "prdn_commitment_changes"),
    machines_where(is_past("Production Commitment Revise Date"), "prdn_overdue"),
    machines_where(
        is_blank("Container Placement date")
        | (is_past("Container Placement date") & is_blank("Container Revision Date")),
        "container_machines_pending",
    ),
    average("no of comm container changes", "container_commitment_changes"),
    machines_where(is_past("Container Revision Date"), "container_overdue"),
    machines_where(status.isin(["pending", "", "nan", "none"]), "clearance_pending"),
], axis=1)

cols = list(mine.columns)
cmp = engine[cols].reindex(mine.index)
mismatch = (mine[cols].fillna(0) != cmp[cols].fillna(0))
print("MATCH" if not mismatch.values.any() else "MISMATCH")
mine